# 📦 The model shelf: from a trained model to an API

A model in a notebook helps nobody else. On the **model shelf** you upload a trained model and it becomes a
**prediction API** in seconds, with its own documentation page. Then a Streamlit app, a script or an **n8n
workflow** can ask it for predictions.

```
your notebook ──pack()──▶  my-model.zip  ──upload──▶  https://models.automate.business.aau.dk
                                                      ├─ POST /m/my-model/predict   ← n8n, Python, apps
                                                      └─ /m/my-model/docs           ← try it in the browser
```

**What works:** any scikit-learn **pipeline** (preprocessing + model): linear and logistic regression, decision
trees, random forests, gradient boosting, XGBoost, for classification and regression.

**How:** the pipeline is converted to **ONNX**, a standard format for trained models that runs without Python code.
That is why the shelf is safe to share: an upload can never run code on the server (a pickle could).

This notebook: 1 · setup → 2 · data → 3 · three models, packed and uploaded → 4 · a regression → 5 · call your API
→ 6 · your own model.

---
## 1 · Setup

The next cell writes the packaging helper, `shelf_pack.py`, next to this notebook. You don't need to read it; it is there so you can.

In [ ]:
%%writefile shelf_pack.py
"""Package a fitted scikit-learn pipeline for the model shelf.

    pack(pipe, X_train, "my-model", description="...", target="...")   ->  my-model.zip, ready to upload

Works for any scikit-learn pipeline (preprocessing + model: linear or logistic regression, trees, forests,
gradient boosting, ...), and for pipelines that end in XGBoost. No pickles: the pipeline is converted to ONNX, a
standard format that runs anywhere without Python code. XGBoost keeps its own booster.json next to the ONNX
preprocessing.
"""
import json
import zipfile

import numpy as np
import pandas as pd
import onnxruntime as ort
from skl2onnx import to_onnx
from sklearn.base import is_classifier
from scipy import sparse
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

OPSET = {"": 17, "ai.onnx.ml": 3}
NUMPY = {"tensor(float)": np.float32, "tensor(double)": np.float64, "tensor(int64)": np.int64,
         "tensor(int32)": np.int32, "tensor(bool)": np.bool_}


def _typed(X, dtype=np.float32):
    """Numbers as floats, everything else as text: the types the shelf feeds the model."""
    X = X.copy()
    for c in X.columns:
        X[c] = X[c].astype(dtype) if pd.api.types.is_numeric_dtype(X[c]) else X[c].astype(str)
    return X


def _double_imputers(onx):
    """onnxruntime has no float64 Imputer; write each one as Where(IsNaN(x), fill, x) instead."""
    from onnx import helper, numpy_helper
    g = onx.graph
    for i, node in reversed(list(enumerate(g.node))):
        if node.op_type != "Imputer":
            continue
        fill = [a for a in node.attribute if a.name == "imputed_value_floats"][0].floats
        const = numpy_helper.from_array(np.array([list(fill)], dtype=np.float64), node.output[0] + "_fill")
        g.initializer.append(const)
        isnan = helper.make_node("IsNaN", [node.input[0]], [node.output[0] + "_nan"])
        where = helper.make_node("Where", [node.output[0] + "_nan", const.name, node.input[0]], [node.output[0]])
        del g.node[i]
        g.node.insert(i, where)
        g.node.insert(i, isnan)
    return onx


def _feed(session, X):
    feed = {}
    for i in session.get_inputs():
        v = X[i.name].to_numpy()
        feed[i.name] = (v.astype(object) if i.type == "tensor(string)" else v.astype(NUMPY[i.type])).reshape(-1, 1)
    return feed


def _categories(pipe):
    """For each text column: the values seen in training, and what scikit-learn does with a new value."""
    def fitted(obj):                       # the fitted steps (a ColumnTransformer keeps them in transformers_)
        yield obj
        for _, step, *_ in getattr(obj, "steps", None) or getattr(obj, "transformers_", None) or []:
            yield from fitted(step)

    out = {}
    for enc in fitted(pipe):
        if not isinstance(enc, OneHotEncoder) or not hasattr(enc, "feature_names_in_"):
            continue
        rare = getattr(enc, "infrequent_categories_", None) or [None] * len(enc.categories_)
        for col, cats, r in zip(enc.feature_names_in_, enc.categories_, rare):
            out[str(col)] = {"known": [str(v) for v in cats][:500],
                             "unknown_to": str(r[0]) if r is not None and len(r) and enc.handle_unknown != "error" else None,
                             "ignore": enc.handle_unknown != "error"}
    return out


def pack(pipe, X, name, description="", target="", folder="."):
    if not isinstance(pipe, Pipeline):
        raise TypeError("Put your model in a Pipeline (for example make_pipeline(preprocessing, model)).")
    columns = list(getattr(pipe, "feature_names_in_", X.columns))
    X = X[columns]
    for c in columns:
        if not pd.api.types.is_numeric_dtype(X[c]) and X[c].isna().any():
            print(f"⚠️  '{c}' has missing values. Fill them with a word like 'unknown' before training.")
    sample = _typed(X.head(1))
    last = pipe.steps[-1][1]
    is_xgb = type(last).__name__.startswith("XGB")
    classifier = is_classifier(last)
    files = {}

    if is_xgb:
        if len(pipe.steps) < 2:
            raise TypeError("Put the preprocessing in front of XGBoost in the pipeline.")
        # float64, like scikit-learn: XGBoost's split points sit exactly on data values, float32 rounding flips rows
        prep = to_onnx(pipe[:-1], _typed(X.head(1), np.float64), target_opset=OPSET)
        files["prep.onnx"] = _double_imputers(prep).SerializeToString()
        files["booster.json"] = bytes(last.get_booster().save_raw("json"))
        # XGBoost reads the zeros of a sparse matrix (e.g. from OneHotEncoder) as "missing": the shelf does the same
        zeros_missing = sparse.issparse(pipe[:-1].transform(X.head(5)))
    else:
        opts = {"zipmap": False} if classifier else None
        try:
            files["model.onnx"] = to_onnx(pipe, sample, options=opts, target_opset=OPSET).SerializeToString()
        except Exception as e:
            raise RuntimeError(f"{type(last).__name__} could not be converted for the shelf ({type(e).__name__}). Try "
                               "another model type: LogisticRegression, RandomForestClassifier, GradientBoostingClassifier "
                               "or XGBoost work. (HistGradientBoosting does not, yet.)") from None

    example = {c: (None if pd.isna(v) else v.item() if hasattr(v, "item") else v) for c, v in X.iloc[0].items()}
    meta = {"name": name, "description": description, "target": target,
            "task": "classification" if classifier else "regression",
            "classes": [c.item() if hasattr(c, "item") else c for c in last.classes_] if classifier else None,
            "columns": columns, "example": example, "zeros_missing": bool(is_xgb and zeros_missing),
            "categories": _categories(pipe)}
    files["shelf.json"] = json.dumps(meta, indent=1).encode()

    # check: does the packaged model give the same answers as the pipeline?
    rows = _typed(X.head(300), np.float64 if is_xgb else np.float32)
    if is_xgb:
        s = ort.InferenceSession(files["prep.onnx"])
        z = s.run(None, _feed(s, rows))[0]
        if meta["zeros_missing"]:
            z[z == 0] = np.nan
        out = last.get_booster().inplace_predict(z)
        got = (np.column_stack([1 - out, out]) if out.ndim == 1 else out) if classifier else out
    else:
        s = ort.InferenceSession(files["model.onnx"])
        res = s.run(None, _feed(s, rows))
        got = res[1] if classifier else res[0].ravel()
    ref = pipe.predict_proba(X.head(300)) if classifier else pipe.predict(X.head(300))
    diff = float(np.abs(np.asarray(got, float) - np.asarray(ref, float)).max())
    scale = 1 if classifier else float(np.abs(ref).max() or 1)
    print(f"{'✅' if diff / scale < 0.02 else '⚠️ '} packaged model vs your pipeline: largest difference {diff:.4g}")

    path = f"{folder}/{name}.zip"
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        for f, b in files.items():
            z.writestr(f, b)
    print(f"📦 {path}: {', '.join(files)}. Upload it to the model shelf.")
    return path

In [ ]:
%pip install -q skl2onnx onnxruntime
import numpy as np, pandas as pd, requests, xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from shelf_pack import pack

SHELF = "https://models.automate.business.aau.dk"
CLASS_CODE = ""        # ✏️ the class code from Moodle
ME = "xx"              # ✏️ your initials or group name: model names must be unique on the shelf


def upload(path, name, description="", edit_key=""):
    """Put a packed model on the shelf (or replace yours, with its edit key). Same as the upload page."""
    with open(path, "rb") as f:
        r = requests.post(f"{SHELF}/api/upload", files={"file": f},
                          data={"name": name, "class_code": CLASS_CODE, "description": description, "edit_key": edit_key})
    d = r.json()
    if not r.ok:
        print("❌", d["detail"])
        return None
    print(f"✅ {d['name']} is on the shelf ({d['task']})\n   endpoint: {d['endpoint']}\n   docs:     {d['docs']}")
    if d["edit_key"]:
        print(f"   🔑 edit key (keep it, shown once): {d['edit_key']}")
    return d

---
## 2 · Data: hotel bookings again

The bookings from session 10, with a few features that are known **when the booking is made**. Few columns on
purpose: whoever calls your API has to send all of them.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv")
df = df[df["adr"].between(0, 1000)].copy()
df["children"] = df["children"].fillna(0)
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
arrival = pd.to_datetime(df["arrival_date_year"].astype(str) + "-" + df["arrival_date_month"] + "-"
                         + df["arrival_date_day_of_month"].astype(str))

NUM = ["lead_time", "adr", "total_nights", "total_guests", "previous_cancellations"]
CAT = ["hotel", "market_segment", "deposit_type", "customer_type"]
train, test = arrival < "2017-01-01", arrival >= "2017-01-01"     # learn from the past, check on the future
X, y = df[NUM + CAT], df["is_canceled"]
X.head()

**The rules for a pipeline that can go on the shelf**
- It is a `Pipeline` that takes the **raw columns** of a pandas DataFrame (the columns the caller will send).
- All preprocessing is **inside** the pipeline (imputer, scaler, one-hot encoder, ...). Anything you compute with
  pandas beforehand, the caller has to compute too.
- Text columns have **no missing values** (fill them with a word like `"unknown"`), and there are no custom
  Python functions in the pipeline (`FunctionTransformer`).

In [ ]:
def prep():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), CAT)])

---
## 3 · Three classifiers: train, pack, upload

Will a booking be cancelled? Train each model, check it on 2017, pack it, and put it on the shelf.

In [ ]:
models = {
    "logit":  LogisticRegression(max_iter=2000),
    "forest": RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=20, n_jobs=-1, random_state=1),
    "xgb":    xgb.XGBClassifier(n_estimators=300, learning_rate=0.08, max_depth=5),
}
fitted = {}
for kind, model in models.items():
    pipe = make_pipeline(prep(), model).fit(X[train], y[train])
    auc = roc_auc_score(y[test], pipe.predict_proba(X[test])[:, 1])
    print(f"\n{kind}: test AUC {auc:.3f}")
    fitted[kind] = pipe
    path = pack(pipe, X[train], f"{ME}-hotel-{kind}", description=f"Will a hotel booking be cancelled? {kind}, AUC {auc:.2f}",
                target="is_canceled")
    upload(path, f"{ME}-hotel-{kind}")

`pack()` checks that the packed model gives the same answers as your pipeline (✅), then writes a `.zip`.
`upload()` does what the upload page on the shelf does. You can also download the zip (folder icon on the left)
and upload it there by hand.

**Open one of the `docs` links.** That page is made for your model: its columns, an example, and *Try it out*.

---
## 4 · A regression: the price of a night

The same works for numbers. Predict the average daily rate (`adr`) from the other columns.

In [ ]:
NUM_P = ["lead_time", "total_nights", "total_guests"]
price = make_pipeline(
    ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM_P),
                       ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)]),
    LinearRegression())
Xp = df[NUM_P + CAT]
price.fit(Xp[train], df.loc[train, "adr"])
print(f"test mean absolute error: {mean_absolute_error(df.loc[test, 'adr'], price.predict(Xp[test])):.1f} EUR")
_ = upload(pack(price, Xp[train], f"{ME}-hotel-price", target="adr"), f"{ME}-hotel-price",
       description="Average price per night (EUR), linear regression")

---
## 5 · Call your API

From Python it is one request. Send one or more rows with **all** the columns; you get one prediction per row.

In [ ]:
booking = {"lead_time": 120, "adr": 95.0, "total_nights": 3, "total_guests": 2, "previous_cancellations": 0,
           "hotel": "City Hotel", "market_segment": "Online TA", "deposit_type": "No Deposit", "customer_type": "Transient"}
r = requests.post(f"{SHELF}/m/{ME}-hotel-xgb/predict", json={"rows": [booking]})
r.json()

In [ ]:
# many rows at once (up to 1,000): the first ten test bookings, next to what really happened
rows = X[test].head(10).to_dict("records")
answer = requests.post(f"{SHELF}/m/{ME}-hotel-forest/predict", json={"rows": rows}).json()["predictions"]
pd.DataFrame({"probability of cancelling": [a["probability"] for a in answer], "cancelled": y[test].head(10).values})

**From n8n:** add an **HTTP Request** node.

| Setting | Value |
|---|---|
| Method | `POST` |
| URL | `https://models.automate.business.aau.dk/m/<your-model>/predict` |
| Send Body | on · Body Content Type **JSON** · Specify Body **Using JSON** |
| JSON | `{{ { rows: [ $json ] } }}` (the incoming item must have all the columns) |

The answer is in `{{ $json.predictions[0].probability }}` (classification) or `{{ $json.predictions[0].prediction }}`.

**From Streamlit:** the same `requests.post(...)` as above. Your app then needs no model files at all.

---
## 6 · Your own model ✏️

Put any model you trained in this course on the shelf: build it as a pipeline on the raw columns, then `pack` and
`upload`. To **replace** it later (a better model, same name), pass the edit key you got:
`upload(path, name, edit_key="...")`. The endpoint stays the same, so apps and workflows keep working.

In [ ]:
# my_pipe = make_pipeline(prep, model).fit(X_train, y_train)
# upload(pack(my_pipe, X_train, f"{ME}-my-model", description="what it predicts, from what"), f"{ME}-my-model")

**What the shelf will not take:** pickles or joblib files, custom Python functions in the pipeline, text or image
models. Files up to 25 MB (a random forest with very deep trees can get larger: limit `max_depth`).